# Pyomo: persistent solves and explicit checks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/pyomo-repeated-solves.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/pyomo-repeated-solves.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup
Use the installed scientific libraries and install only missing dependencies. The shared helper keeps this routine code in one place. Solver-specific lessons introduce additional solvers at the point where they are used.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'highspy': 'highspy', 'pyomo': 'pyomo'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = "appsi_highs"
assert pyo.SolverFactory(SOLVER).available(exception_flag=False)


## Reuse a model when the data change
This small example demonstrates a mutable parameter and the APPSI HiGHS interface. The solver interface detects relevant changes between solves. Reuse is useful for sensitivity experiments, but you must still check termination and validate the returned values after each change.

The collection targets Pyomo 6.10.1. It uses the documented APPSI interface rather than silently mixing the different result objects of the newer development-preview solver APIs. See [Pyomo's APPSI documentation](https://pyomo.readthedocs.io/en/stable/reference/topical/appsi/appsi.html).


In [ ]:
from pyomo.contrib.appsi.solvers import Highs
from pyomo.contrib.appsi.base import TerminationCondition
m = pyo.ConcreteModel('Resource experiment')
m.capacity = pyo.Param(initialize=10,mutable=True)
m.x = pyo.Var(domain=pyo.NonNegativeReals)
m.profit = pyo.Objective(expr=3*m.x,sense=pyo.maximize)
m.limit = pyo.Constraint(expr=2*m.x <= m.capacity)
solver = Highs()
observations = []
for capacity in [10,12,8]:
    m.capacity.set_value(capacity)
    result = solver.solve(m)
    assert result.termination_condition == TerminationCondition.optimal
    assert abs(pyo.value(m.x)-capacity/2) < 1e-7
    observations.append((capacity,pyo.value(m.x),pyo.value(m.profit)))
observations


## Failed solves must not look like solutions
Do not load a solution until termination has been checked. With `load_solution=False`, the interface can report infeasibility without trying to load nonexistent values. Never display values left over from an earlier successful solve as if they solved the new model.


In [ ]:
m.capacity.set_value(-1)
solver.config.load_solution = False
result = solver.solve(m)
assert result.termination_condition == TerminationCondition.infeasible
print('Correctly detected infeasible model; previous variable values are not a solution.')


## Your experiment
Restore a feasible capacity and explicitly re-enable solution loading. Predict the objective before re-solving. Explain why a mutable parameter is useful and when rebuilding the model would be clearer.
